# Лабораторная 2. Где живут данные

Отчёт из первой работы понравился, и через неделю к вам приходят снова.
Аналитик просит добавить платежи, потом категории товаров, потом штат
покупателя. Вы честно дописываете `merge` за `merge`; ноутбук растёт до трёхсот
строк, считается полторы минуты и читается только вами. А потом приходит второй
аналитик и говорит: «дай мне доступ, я сам посчитаю».

И он прав. Расчёт, который умеет запускать один человек, – это не система,
а человек, который однажды уйдёт в отпуск. Сегодня мы переложим данные туда, где их смогут читать разные
люди разными запросами, и разберёмся, что происходит с числами, когда таблицы
соединяют между собой.

## Перед началом: новые слова

| Термин | Простыми словами |
|---|---|
| **Схема таблицы – schema** | Описание колонок и их типов, которое хранится вместе с данными. |
| **Строковое хранение – row-oriented** | На диске рядом лежат значения одной строки. |
| **Колоночное хранение – columnar** | На диске рядом лежат значения одной колонки. |
| **Parquet** | Колоночный формат файла, который хранит и данные, и их схему. |
| **OLTP и OLAP** | Две роли базы: обслуживать операции приложения или отвечать на вопросы к данным. |
| **Подзапрос с именем – CTE** | Именованный промежуточный результат внутри запроса, блок `WITH ... AS`. |
| **Оконная функция – window function** | Агрегат, который считается вокруг строки и не схлопывает её. |

## Подготовка

In [ ]:
# Готовим окружение: в VS Code и Jupyter берём открытую папку курса,
# в Colab материалы скачиваются один раз. Всё остальное – в src/notebook_env.py.
from pathlib import Path
import subprocess, sys

ROOT = next((c for s in (Path.cwd().resolve(), Path("/content/data-engineering-ai-course"))
             for c in (s, *s.parents) if (c / ".course-root").exists()), None)
if ROOT is None:                                   # чистая сессия Colab
    url = ''
    if not url:
        raise RuntimeError("Откройте папку курса в VS Code/Jupyter или склонируйте "
                           "репозиторий в /content/data-engineering-ai-course")
    subprocess.run(["git", "clone", "--depth", "1", url,
                    "/content/data-engineering-ai-course"], check=True)
    ROOT = Path("/content/data-engineering-ai-course")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from notebook_env import setup_course
setup_course(ROOT)


In [ ]:
import time

import duckdb
import pandas as pd
import paths
from course_visuals import show_external_visual, show_visual
from get_data import get_olist

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

RAW = get_olist()
print("duckdb:", duckdb.__version__)

## Как таблица лежит на диске

Данные в конечном счёте лежат в файлах – вопрос в том, как именно значения разложены
внутри. Вариантов два, и от выбора зависит цена запроса.

In [ ]:
show_visual("storage")

CSV устроен по-строчному: строка файла – строка таблицы. Чтобы получить
среднюю сумму заказа, придётся прочитать и разобрать все колонки, включая
адреса и комментарии, которые в расчёте не участвуют.

Колоночные форматы хранят рядом значения одной колонки. Аналитический запрос
обычно трогает три-четыре колонки из сорока, поэтому читать нужно только их.
Побочный эффект – сжатие: в одной колонке значения однотипны и часто
повторяются, так что они ужимаются лучше, чем вперемешку.

Самый распространённый такой формат – **Parquet**. Кроме данных он хранит
схему: какие колонки, каких типов. Тот, кто прочитает файл следующим, получит
дату датой, а не текстом.

Наша схема выше показывает идею. А как она выглядит внутри настоящего файла?

In [ ]:
show_external_visual(
    "parquet_layout",
    caption="Файл Parquet делится на row groups, внутри них данные лежат по колонкам.")

Файл разбит на горизонтальные куски – row groups, по несколько десятков тысяч
строк в каждом. Внутри куска данные разложены по колонкам, а в footer записано,
где именно лежит каждая из них. Поэтому движок читает только те колонки и те
куски, которые нужны запросу, а остальное даже не трогает. Остальные детали
схемы – страницы, кодировки, устройство метаданных – сейчас не нужны.

Переложим наши файлы в Parquet и посмотрим на размеры.

In [ ]:
CSV_TO_PARQUET = [
    "olist_orders_dataset", "olist_order_items_dataset",
    "olist_order_payments_dataset", "olist_order_reviews_dataset",
    "olist_customers_dataset", "olist_products_dataset",
    "olist_sellers_dataset", "olist_geolocation_dataset",
    "product_category_name_translation",
]

rows = []
for name in CSV_TO_PARQUET:
    csv_path = RAW / f"{name}.csv"
    pq_path = paths.PARQUET / f"{name}.parquet"
    df = pd.read_csv(csv_path)
    df.to_parquet(pq_path, index=False)
    rows.append({
        "таблица": name.replace("olist_", "").replace("_dataset", ""),
        "строк": len(df),
        "CSV, МБ": round(csv_path.stat().st_size / 1024 ** 2, 2),
        "Parquet, МБ": round(pq_path.stat().st_size / 1024 ** 2, 2),
    })

sizes = pd.DataFrame(rows)
sizes["во сколько раз меньше"] = (sizes["CSV, МБ"] / sizes["Parquet, МБ"]).round(1)
sizes

Выигрыш в размере зависит от данных. На таблицах, где почти всё – уникальные
тридцатидвухсимвольные идентификаторы, он скромный: сжимать нечего.
На географии, где повторяются штаты и города, разница заметнее. В производственных
таблицах с датами, категориями и числами Parquet обычно выигрывает в разы.

Интереснее посмотреть не на размер, а на цену чтения. Возьмём две колонки из
самого большого файла.

In [ ]:
big_csv = RAW / "olist_geolocation_dataset.csv"
big_pq = paths.PARQUET / "olist_geolocation_dataset.parquet"
cols = ["geolocation_zip_code_prefix", "geolocation_state"]

t0 = time.perf_counter()
via_csv = pd.read_csv(big_csv, usecols=cols)
t_csv = time.perf_counter() - t0

t0 = time.perf_counter()
via_parquet = pd.read_parquet(big_pq, columns=cols)
t_pq = time.perf_counter() - t0

print(f"строк прочитано      : {len(via_csv):,} из CSV и {len(via_parquet):,} из Parquet")
print(f"две колонки из CSV   : {t_csv:6.2f} c")
print(f"две колонки из Parquet: {t_pq:6.2f} c")

Мы просили две колонки в обоих случаях, но CSV пришлось прочитать целиком
и разобрать все пять колонок, чтобы выбросить три. Parquet прочитал ровно два
нужных куска файла.

Отсюда практика, принятая почти везде: исходные файлы остаются в том виде,
в каком приехали, а рядом лежит их колоночная копия, с которой работают все
расчёты.

## Две разные жизни одной базы

Базу данных тоже выбирают по характеру запросов. Магазину нужно быстро достать
один заказ по номеру, записать новый и не потерять его при сбое – это
**OLTP**, обработка транзакций. Аналитику нужно просуммировать миллион строк
по трём колонкам за два года – это **OLAP**, аналитические запросы.

| | OLTP | OLAP |
|---|---|---|
| Типичный запрос | «покажи заказ №12345 и обнови статус» | «выручка по категориям за два года» |
| Строк за запрос | единицы | миллионы |
| Колонок за запрос | все | три-четыре из сорока |
| Хранение | построчное | колоночное |
| Примеры | PostgreSQL, MySQL | ClickHouse, BigQuery, DuckDB |

Поэтому аналитику и не дают ключи от базы магазина: один задумчивый запрос
по всей истории заказов – и магазин перестаёт принимать новые. Такие истории
заканчиваются не выговором аналитику, а отдельным хранилищем для аналитики.

Мы возьмём **DuckDB**: аналитический движок, который работает прямо внутри
процесса Python, без отдельного сервера. Запросы, которые мы напишем, почти
дословно переносятся в PostgreSQL или ClickHouse.

In [ ]:
con = duckdb.connect(str(paths.WAREHOUSE))

RAW_TABLES = {
    "raw_orders": "olist_orders_dataset",
    "raw_items": "olist_order_items_dataset",
    "raw_payments": "olist_order_payments_dataset",
    "raw_reviews": "olist_order_reviews_dataset",
    "raw_customers": "olist_customers_dataset",
    "raw_products": "olist_products_dataset",
    "raw_sellers": "olist_sellers_dataset",
    "raw_category_translation": "product_category_name_translation",
}

for table, file in RAW_TABLES.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT * FROM read_parquet('{(paths.PARQUET / (file + ".parquet")).as_posix()}')
    """)

con.execute("SHOW TABLES").df()

`CREATE OR REPLACE` здесь не случайность: запустив ячейку второй раз, вы
получите тот же результат, а не удвоенные данные. Это идемпотентность из первой
работы, выраженная средствами SQL.

In [ ]:
con.execute("DESCRIBE raw_orders").df()

Даты приехали как `VARCHAR`. Формат сохранил ровно те типы, которые ему дали:
мы прочитали CSV, где всё было текстом, и записали текст. Формат хранит схему,
но не додумывает её. Приведением типов займёмся в третьей работе, а пока
будем преобразовывать их прямо в запросе.

## Спрашиваем данные на SQL

SQL описывает, что нужно получить, а не как это посчитать: движок сам решит,
в каком порядке читать и что держать в памяти. Порядок, в котором он логически
выполняет запрос, отличается от порядка написания:

```
  FROM / JOIN  →  WHERE  →  GROUP BY  →  HAVING  →  SELECT  →  ORDER BY  →  LIMIT
```

Отсюда два следствия, на которых спотыкаются регулярно: `WHERE` отбирает строки
до группировки, а `HAVING` – агрегаты после неё; и псевдоним из `SELECT` нельзя
использовать в `WHERE`, потому что к моменту отбора его ещё не существует.

In [ ]:
con.execute("""
    SELECT order_id, order_status,
           CAST(order_purchase_timestamp AS TIMESTAMP) AS purchased_at
    FROM raw_orders
    WHERE order_status = 'canceled'
    ORDER BY purchased_at DESC
    LIMIT 5
""").df()

In [ ]:
con.execute("""
    SELECT order_status,
           COUNT(*) AS n_orders,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM raw_orders
    GROUP BY order_status
    HAVING COUNT(*) > 100
    ORDER BY n_orders DESC
""").df()

## Что происходит при соединении

Это главный сюжет работы. В прошлый раз мы видели, как заказы размножаются
позициями. Теперь посмотрим на все варианты сразу и добавим случай, который
ломает отчёты чаще всего.

In [ ]:
show_visual("join")

In [ ]:
con.execute("""
    SELECT 'заказы → покупатели (1:1)' AS связь,
           (SELECT COUNT(*) FROM raw_orders) AS строк_слева,
           (SELECT COUNT(*) FROM raw_orders o JOIN raw_customers c USING (customer_id))
               AS после_соединения
    UNION ALL
    SELECT 'заказы → платежи (1:N)',
           (SELECT COUNT(*) FROM raw_orders),
           (SELECT COUNT(*) FROM raw_orders o JOIN raw_payments p USING (order_id))
    UNION ALL
    SELECT 'заказы → позиции (1:N)',
           (SELECT COUNT(*) FROM raw_orders),
           (SELECT COUNT(*) FROM raw_orders o JOIN raw_items i USING (order_id))
    UNION ALL
    SELECT 'заказы → позиции → платежи',
           (SELECT COUNT(*) FROM raw_orders),
           (SELECT COUNT(*) FROM raw_orders o
              JOIN raw_items i USING (order_id)
              JOIN raw_payments p USING (order_id))
""").df()

Первая строка безопасна: у заказа один покупатель, количество не изменилось.
Вторая и третья дают знакомое размножение. А вот четвёртая – это уже другой
масштаб бедствия: соединив заказ одновременно с позициями и платежами, мы
получили все сочетания внутри каждого заказа. Три позиции и два платежа дают
шесть строк, и сумма платежей в таком отчёте втрое больше настоящей. Выручка,
выросшая от одного лишнего соединения, – это ровно тот случай, когда цифру
радостно уносят на совещание и обнаруживают ошибку через месяц.

Лечение одно и работает во всех случаях: сначала свернуть правую таблицу до
зерна левой, потом соединять. Удобный инструмент для этого – **подзапрос
с именем (CTE)**: он не ускоряет запрос, а разбивает его на читаемые шаги.

In [ ]:
demo = con.execute("""
    WITH pay AS (
        SELECT order_id,
               SUM(payment_value) AS payment_total,
               COUNT(*)           AS payment_count
        FROM raw_payments
        GROUP BY order_id
    )
    SELECT o.order_id,
           o.order_status,
           COALESCE(p.payment_total, 0) AS payment_total,
           COALESCE(p.payment_count, 0) AS payment_count
    FROM raw_orders o
    LEFT JOIN pay p USING (order_id)
""").df()

total_payments = con.execute("SELECT SUM(payment_value) FROM raw_payments").fetchone()[0]
print(f"строк в результате   : {len(demo):,}")
print(f"заказов в raw_orders : {con.execute('SELECT COUNT(*) FROM raw_orders').fetchone()[0]:,}")
print(f"сумма платежей       : {demo['payment_total'].sum():,.2f}")
print(f"она же в raw_payments: {total_payments:,.2f}")
print(f"заказов без платежей : {(demo['payment_count'] == 0).sum()}")

Количество строк не изменилось, сумма сошлась до копейки, заказ без платежей
остался в результате с нулём вместо пропуска. Этот шаблон – свёртка в CTE,
затем `LEFT JOIN`, затем `COALESCE` – дальше в курсе встретится ещё много раз.

## Агрегат, который не схлопывает строки

`GROUP BY` превращает много строк в одну. Но иногда нужно оставить строки на
месте и добавить к каждой контекст: сколько всего накопилось к этому месяцу,
сколько было в предыдущем, какое место занимает продавец в своём штате. Это
делают **оконные функции**.

In [ ]:
con.execute("""
    WITH monthly AS (
        SELECT date_trunc('month', CAST(order_purchase_timestamp AS TIMESTAMP)) AS month,
               COUNT(*) AS n_orders
        FROM raw_orders
        GROUP BY 1
    )
    SELECT month, n_orders,
           LAG(n_orders) OVER (ORDER BY month) AS prev_month,
           SUM(n_orders) OVER (ORDER BY month ROWS UNBOUNDED PRECEDING) AS cumulative
    FROM monthly
    ORDER BY month
    LIMIT 8
""").df()

Без окон такие вещи считают соединением таблицы с самой собой – запрос
получается длиннее и медленнее. Инженеру они нужны постоянно: накопительные
итоги, сравнение с прошлым периодом, «последняя версия записи по ключу»,
отбраковка дубликатов.

## Что ломается в этом месте

**Условие по правой таблице отменило `LEFT JOIN`.**

In [ ]:
q = """
    SELECT COUNT(*) FROM raw_orders o
    LEFT JOIN raw_payments p USING (order_id)
    {cond}
"""
cond_card = "WHERE p.payment_type = 'credit_card'"
cond_with_null = cond_card + " OR p.payment_type IS NULL"

print(f"без условия                      : {con.execute(q.format(cond='')).fetchone()[0]:>8,}")
print(f"условие по правой таблице        : {con.execute(q.format(cond=cond_card)).fetchone()[0]:>8,}")
print(f"то же условие с учётом пропусков : {con.execute(q.format(cond=cond_with_null)).fetchone()[0]:>8,}")

`LEFT JOIN` добавил заказы без платежей, подставив им пропуски, а следующий за
ним `WHERE` эти пропуски отбросил. Соединение осталось левым только на бумаге.
Если условие относится к самому соединению, его место в `ON`, а не в `WHERE`.

**Посчитали количество строк вместо количества заказов.**

In [ ]:
con.execute("""
    SELECT COUNT(*)                   AS строк_после_соединения,
           COUNT(DISTINCT o.order_id) AS заказов,
           ROUND(AVG(i.price), 2)     AS средняя_цена_позиции
    FROM raw_orders o
    JOIN raw_items i USING (order_id)
""").df()

Разница между первыми двумя числами – те самые размноженные строки. Средняя
цена позиции здесь посчитана корректно, потому что это показатель уровня
позиции. А вот средний чек по этой таблице считать нельзя: заказы с большим
числом позиций получат больший вес.

## Задание 1. Одна строка – один заказ

Соберите таблицу `orders_enriched`, где одна строка соответствует одному заказу:

| колонка | смысл |
|---|---|
| `order_id`, `customer_id`, `order_status` | из таблицы заказов |
| `order_date` | дата покупки, тип `DATE` |
| `payment_total`, `payment_count` | сумма и количество платёжных операций |
| `n_items`, `items_total`, `freight_total` | количество позиций, сумма товаров и доставки |

Условия:

1. Строк ровно столько, сколько заказов, `order_id` уникален.
2. Сумма `payment_total` по всей таблице совпадает с суммой платежей
   в `raw_payments`, сумма `items_total` – с суммой цен в `raw_items`. Если
   какая-то из сумм больше, значит строки размножились.
3. Заказы без платежей и заказы без позиций остаются в таблице с нулями.
4. Пропусков в числовых колонках нет.

Подумайте, сколько промежуточных свёрток вам нужно и почему нельзя соединить
заказы, позиции и платежи одним запросом.

In [ ]:
# Структура подсказана выше: сначала свёртки в CTE, затем соединения.
sql_task1 = """
CREATE OR REPLACE TABLE orders_enriched AS
SELECT 1
"""

con.execute(sql_task1)
con.execute("SELECT * FROM orders_enriched LIMIT 5").df()

In [ ]:
from checks.lab02 import check_task_1, check_task_2, lab02_result

check_task_1(con)

## Задание 2. Динамика по месяцам

Создайте таблицу `monthly_revenue` – одна строка на календарный месяц:

| колонка | смысл |
|---|---|
| `month` | первый день месяца, тип `DATE` |
| `n_orders` | заказов за месяц |
| `revenue` | выручка за месяц, сумма `items_total` |
| `revenue_cumulative` | накопительная выручка с начала истории |
| `revenue_prev` | выручка предыдущего месяца, для первого – пропуск |

Условия:

1. Считаем от `orders_enriched`, а не от сырых таблиц.
2. Накопительный итог и предыдущий месяц получены оконными функциями.
3. В последнем месяце накопительный итог равен всей выручке.

In [ ]:
sql_task2 = """
CREATE OR REPLACE TABLE monthly_revenue AS
SELECT 1
"""

con.execute(sql_task2)
con.execute("SELECT * FROM monthly_revenue LIMIT 5").df()

In [ ]:
check_task_2(con, sql_task2)

Напоследок выгрузим таблицу из хранилища в файл. Так делают, когда результат
нужен тому, у кого нет доступа к базе: смежной команде, стороннему сервису,
соседнему процессу. Файл в колоночном формате прочитает почти любой инструмент,
а лишних прав для этого не потребуется.

In [ ]:
con.execute(f"""
    COPY orders_enriched TO '{(paths.STAGING / "lab02_orders_enriched.parquet").as_posix()}'
    (FORMAT PARQUET)
""")
print("сохранено:", (paths.STAGING / "lab02_orders_enriched.parquet").relative_to(paths.ROOT))

## Та же задача в другой системе

Сорок филиалов городской библиотеки. Каждую ночь филиал выгружает CSV с
выдачами и возвратами за день: суммарно около 40 МБ в сутки, примерно триста
тысяч строк. Нужно три вещи: ежемесячный отчёт в мэрию, поиск «что у читателя
на руках сейчас» для библиотекаря и хранение истории за десять лет.

Коллега предлагает: брокер событий для сбора выгрузок, распределённый движок
для обработки, колоночная СУБД для отчётов.

Ответьте письменно: какие здесь источники и какого типа данные; пакетная
обработка или потоковая; сколько места займёт история за десять лет; нужны ли
три предложенных компонента, каждый отдельно; запрос библиотекаря – это OLTP
или OLAP и из какой базы его обслуживать; какие риски у ночных выгрузок;
как выглядит ваша архитектура в трёх-четырёх компонентах.

### Ваш ответ

1. Источники и тип данных: …
2. Пакетная обработка или потоковая: …
3. Оценка объёма за десять лет: …
4. Брокер, распределённый движок, колоночная СУБД: …
5. Запрос библиотекаря: …
6. Риски ночных выгрузок: …
7. Итоговая архитектура: …

In [ ]:
lab02_result()
con.close()

## Вопросы для проверки себя

1. У вас таблица на двести колонок и запрос, которому нужны три. Почему
   колоночный формат прочитает его быстрее, если оба файла лежат на одном диске?
2. Почему `LEFT JOIN` с условием по правой таблице в `WHERE` ведёт себя как
   обычное соединение? Как записать то же условие, сохранив смысл?
3. Вам нужно среднее число позиций в заказе. Почему нельзя взять `AVG` по
   таблице, полученной соединением заказов с позициями?
4. Чем оконная функция отличается от `GROUP BY`? Приведите задачу, которую без
   окна пришлось бы решать соединением таблицы с самой собой.
5. Компания хранит 80 ГБ аналитических данных, запросов немного, бюджет
   ограничен. Нужна ли здесь отдельная колоночная СУБД? Что вы спросите
   у заказчика, прежде чем отвечать?

## Что дальше

Теперь данные лежат в хранилище, их можно читать на SQL, а соединения не портят
цифры. Но считаем мы всё ещё по данным, которые никто не проверял: даты лежат
текстом, у части заказов статус «доставлен» без даты доставки, один и тот же
отзыв встречается дважды.

В следующей работе разберёмся, как отличить дефект данных от нормы и что
делать со строками, которые не проходят проверку.